# Exploratory Data Analysis (EDA)
## COM747 — Data Science and Machine Learning CW2
**Research title:** A Machine Learning Framework for Detecting Multidimensional Care-Intensity Discordance and Predicting 30-Day Hospital Readmission in Patients with Diabetes.

**Dataset:** [UCI Diabetes 130-US Hospitals for Years 1999–2008](https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008).

**RQ1:** How is care-intensity discordance associated with 30-day hospital readmission after accounting for patient characteristics and recorded clinical complexity?

**RQ2:** Can care-intensity discordance features improve readmission prediction compared with conventional ML models using patient characteristics and observed care variables?

This notebook consolidates the original six plots and the expanded initial EDA. It covers CRISP-DM data understanding, not final cleaning, feature engineering or modelling. Each row is a hospital encounter; a patient may appear more than once. The raw dataset is not modified on disk.

**How to run:** Place this file in `eda/`, put `diabetic_data.csv` in the parent project's `data/raw/` folder, follow the parent README, select **Python (DSML CW2)** as the notebook kernel, then restart and run all cells. The loader works from the project root or `eda/` and also checks for a local CSV. Do not create another virtual environment.

**Analysis boundary:** Full-dataset EDA here is descriptive only. Before model-directed exploration, define the eligible discharge cohort and reserve a patient-grouped test set; fit all transformations and expected-care models using training data only. These plots do not establish causation or appropriate clinical treatment. The dataset is historical and recorded readmissions may not capture every subsequent admission.

## Phase 1 — Initial data understanding

### 1. Import required libraries

Set the visualisation style and verify the shared kernel.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", None)

print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())

### 2. Import the dataset

Preserve literal `None` test-result labels (test not performed) and convert `?` and empty fields to missing values. `df_raw` is the imported source snapshot; `df` is a working copy.

In [ ]:
candidates = [
    Path("../data/raw/diabetic_data.csv"),
    Path("data/raw/diabetic_data.csv"),
    Path("diabetic_data.csv"),
    Path("../diabetic_data.csv"),
]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError(
        "Place diabetic_data.csv in the project data/raw/ directory. "
        f"Current working directory: {Path.cwd()}"
    )

df_raw = pd.read_csv(data_path, keep_default_na=False, na_values=["?", ""])
df = df_raw.copy()
print("Loaded:", data_path.resolve())
display(df.head())

### 3. Initial dataset inspection

Inspect the dimensions, data types and unique patient counts.

In [ ]:
print("Number of hospital encounters:", len(df))
print("Number of columns:", df.shape[1])
print("Number of unique patients:", df["patient_nbr"].nunique())
print("\nDataset information:")
df.info()
print("\nColumn names:", df.columns.tolist())

### 4. Descriptive statistics

Summarise numeric and categorical variables. Identifiers are excluded from numeric summaries because their magnitudes have no clinical meaning.

In [ ]:
display(df.drop(columns=["encounter_id", "patient_nbr"]).describe().T)
display(df.describe(include=["object", "category", "string"]).T)

### 5. Missing-value analysis

Quantify missing data. Literal `None` in `A1Cresult` and `max_glu_serum` is kept separate from missing values.

In [ ]:
missing_values = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": df.isna().mean() * 100,
})
missing_values = missing_values.loc[missing_values["Missing Count"] > 0].sort_values(
    "Missing Percentage", ascending=False
)
display(missing_values.round(2))

if not missing_values.empty:
    plt.figure(figsize=(12, max(4, len(missing_values) * 0.45)))
    sns.barplot(data=missing_values.rename_axis("Feature").reset_index(),
                x="Missing Percentage", y="Feature", color="steelblue")
    plt.title("Missing Values by Dataset Feature")
    plt.xlabel("Missing Values (%)")
    plt.ylabel("Features")
    plt.xlim(0, 100)
    plt.tight_layout()
    plt.show()
else:
    print("No missing values detected under the current import rules.")

### 6. Duplicate records and repeated patients

Repeated patient encounters are not necessarily duplicate records. Patient identifiers will be needed for grouped splitting but should not be used as predictors.

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate encounter IDs:", df["encounter_id"].duplicated().sum())
print("Patients with multiple encounters:", (df["patient_nbr"].value_counts() > 1).sum())

## Phase 2 — Demographic analysis

### 7. Hospital encounter distribution by age

Original EDA plot 1: identify the age groups most represented in the dataset.

In [ ]:
age_counts = df["age"].value_counts().sort_index()
plt.figure(figsize=(12, 6))
age_counts.plot(kind="bar", color="steelblue", edgecolor="black")
plt.title("Distribution of Hospital Encounters by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Number of Hospital Encounters")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
display(age_counts)

### 8. Gender distribution

Original EDA plot 2: summarise recorded gender by encounter.

In [ ]:
gender_counts = df["gender"].fillna("Missing / Unrecorded").value_counts()
plt.figure(figsize=(8, 8))
plt.pie(gender_counts, labels=gender_counts.index, autopct="%1.1f%%",
        startangle=90, wedgeprops={"edgecolor": "white"})
plt.title("Gender Distribution of Hospital Encounters")
plt.axis("equal")
plt.tight_layout()
plt.show()
display(gender_counts)

### 9. Racial distribution

Support later equality, diversity and inclusion analysis. Preserve missing race as an explicit descriptive group.

In [ ]:
race_counts = df["race"].fillna("Missing / Unrecorded").value_counts()
plt.figure(figsize=(10, 6))
race_counts.plot(kind="bar", color="teal", edgecolor="black")
plt.title("Distribution of Hospital Encounters by Race")
plt.xlabel("Recorded Race")
plt.ylabel("Number of Hospital Encounters")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
display(race_counts)

## Phase 3 — Readmission analysis

### 10. Hospital readmission distribution

Original EDA plot 3. The original categories are `NO` (no recorded readmission), `>30` (later readmission) and `<30` (early readmission). The research uses the original `<30` label as its operational 30-day outcome.

In [ ]:
readmission_order = ["NO", ">30", "<30"]
assert df["readmitted"].isin(readmission_order).all(), "Unexpected or missing readmission labels"
readmission_counts = df["readmitted"].value_counts().reindex(readmission_order, fill_value=0)
plt.figure(figsize=(9, 6))
ax = readmission_counts.plot(kind="bar", color=["seagreen", "orange", "crimson"], edgecolor="black")
ax.bar_label(ax.containers[0])
plt.title("Hospital Readmission Distribution")
plt.xlabel("Readmission Category")
plt.ylabel("Number of Hospital Encounters")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
display(readmission_counts)

### 11. Readmission percentage distribution

Original EDA plot 5: show proportions of the three original categories.

In [ ]:
plt.figure(figsize=(8, 8))
plt.pie(readmission_counts, labels=readmission_counts.index, autopct="%1.1f%%",
        startangle=90, colors=["seagreen", "orange", "crimson"], explode=[0, 0, 0.08])
plt.title("Percentage Distribution of Hospital Readmissions")
plt.axis("equal")
plt.tight_layout()
plt.show()

### 12. Binary research target and class imbalance

Map `<30` to 1 and both other valid categories to 0 for descriptive analysis. This is not yet the final modelling cohort. Never use either readmission label as an input feature.

In [ ]:
df["readmitted_30"] = df["readmitted"].map({"<30": 1, ">30": 0, "NO": 0}).astype(int)
counts = df["readmitted_30"].value_counts().reindex([0, 1], fill_value=0)
display(pd.DataFrame({"Encounters": counts, "Percentage": counts / len(df) * 100}).round(2))
plt.figure(figsize=(8, 6))
ax = counts.plot(kind="bar", color="steelblue")
ax.bar_label(ax.containers[0])
plt.title("30-Day Readmission Target Distribution")
plt.xlabel("Readmission Status")
plt.ylabel("Number of Hospital Encounters")
plt.xticks([0, 1], ["No Early Readmission", "Early Readmission"], rotation=0)
plt.tight_layout()
plt.show()

### 13. Readmission percentage by age group

Original EDA plot 6: percentages are calculated within each age group, not across the whole dataset.

In [ ]:
age_readmission = pd.crosstab(df["age"], df["readmitted"], normalize="index").reindex(
    columns=readmission_order, fill_value=0) * 100
display(age_readmission.round(2))
age_readmission.plot(kind="bar", figsize=(14, 6), color=["seagreen", "orange", "crimson"])
plt.title("Readmission Percentage by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Readmission Percentage (%)")
plt.xticks(rotation=45)
plt.legend(title="Readmission Category")
plt.tight_layout()
plt.show()

### 14. 30-day readmission by demographic groups

Compare within-group rates and denominators. Small groups can have unstable rates; these descriptive differences are not evidence of model unfairness.

In [ ]:
for feature in ["age", "gender", "race"]:
    group_labels = df[feature].fillna("Missing / Unrecorded")
    summary = df.groupby(group_labels)["readmitted_30"].agg(
        Encounters="count", Early_Readmissions="sum", Rate="mean")
    summary["Rate"] *= 100
    summary = summary.rename(columns={"Rate": "Readmission Rate (%)"})
    print(f"\n30-Day Readmission by {feature}")
    display(summary.round(2))
    summary["Readmission Rate (%)"].plot(kind="bar", figsize=(10, 5), color="steelblue")
    plt.title(f"30-Day Readmission Rate by {feature.title()}")
    plt.xlabel(feature.title())
    plt.ylabel("30-Day Readmission Rate (%)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Phase 4 — Care-intensity analysis

The four recorded care dimensions are the proposed regression targets for expected-care estimation. At this stage we only describe observed care; no discordance score is calculated yet.

### 15. Identify the four care-intensity dimensions

Hospital stay (days), laboratory procedures, non-laboratory procedures and number of distinct medications administered.

In [ ]:
care_features = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications"]
care_labels = {
    "time_in_hospital": "Hospital Stay Duration",
    "num_lab_procedures": "Laboratory Procedures",
    "num_procedures": "Medical Procedures",
    "num_medications": "Medication Count",
}
display(df[care_features].describe().T)

### 16. Hospital stay duration distribution

Original EDA plot 4: frequency of recorded inpatient days.

In [ ]:
stay_counts = df["time_in_hospital"].value_counts().sort_index()
plt.figure(figsize=(12, 6))
stay_counts.plot(kind="bar", color="orange", edgecolor="black")
plt.title("Distribution of Hospital Stay Duration")
plt.xlabel("Number of Days in Hospital")
plt.ylabel("Number of Hospital Encounters")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 17. Distribution of all four care-intensity variables

Explore variability and skewness.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, feature in zip(axes.flat, care_features):
    sns.histplot(data=df, x=feature, discrete=True, ax=ax, color="steelblue")
    ax.set_title(f"Distribution of {care_labels[feature]}")
    ax.set_xlabel(care_labels[feature])
    ax.set_ylabel("Number of Encounters")
fig.suptitle("Distribution of Multidimensional Care Intensity", fontsize=16)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### 18. Outlier exploration using boxplots

Unusual care patterns may be important. Do not automatically remove valid extreme values.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, feature in zip(axes.flat, care_features):
    sns.boxplot(data=df, y=feature, ax=ax, color="skyblue")
    ax.set_title(f"Boxplot of {care_labels[feature]}")
    ax.set_ylabel(care_labels[feature])
fig.suptitle("Outlier Analysis of Care-Intensity Variables", fontsize=16)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### 19. Care intensity versus 30-day readmission

Unadjusted comparisons provide context for RQ1 but do not establish a relationship independent of clinical complexity.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, feature in zip(axes.flat, care_features):
    sns.boxplot(data=df, x="readmitted_30", y=feature, order=[0, 1],
                hue="readmitted_30", hue_order=[0, 1], palette=["seagreen", "salmon"],
                dodge=False, legend=False, ax=ax)
    ax.set_title(f"{care_labels[feature]} by Readmission")
    ax.set_xlabel("30-Day Readmission Status")
    ax.set_ylabel(care_labels[feature])
    ax.set_xticks([0, 1], ["No", "Yes"])
fig.suptitle("Care-Intensity Comparison by 30-Day Readmission", fontsize=16)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### 20. Statistical summary of care intensity by readmission

Report count, mean, median and standard deviation to complement the plots.

In [ ]:
care_summary = df.groupby("readmitted_30")[care_features].agg(["count", "mean", "median", "std"])
display(care_summary.round(2))

### 21. Readmission rate by hospital stay duration

Show group sizes alongside the rate to help interpret less common stay lengths.

In [ ]:
stay_readmission = df.groupby("time_in_hospital")["readmitted_30"].agg(
    Encounters="count", Early_Readmissions="sum", Rate="mean")
stay_readmission["Rate"] *= 100
stay_readmission = stay_readmission.rename(columns={"Rate": "Readmission Rate (%)"})
display(stay_readmission.round(2))
plt.figure(figsize=(12, 6))
plt.plot(stay_readmission.index, stay_readmission["Readmission Rate (%)"], marker="o", linewidth=2, color="crimson")
plt.title("30-Day Readmission Rate by Hospital Stay Duration")
plt.xlabel("Hospital Stay (Days)")
plt.ylabel("30-Day Readmission Rate (%)")
plt.xticks(stay_readmission.index)
plt.tight_layout()
plt.show()

## Phase 5 — Clinical complexity and previous healthcare utilisation

Recorded diagnosis counts are imperfect proxies for clinical complexity, and may depend on care received during the encounter. They are not verified admission-time severity measures.

### 22. Clinical complexity and utilisation statistics

Inspect recorded diagnosis count and prior-year inpatient, outpatient and emergency utilisation.

In [ ]:
clinical_features = ["number_diagnoses", "number_inpatient", "number_outpatient", "number_emergency"]
display(df[clinical_features].describe().T)

### 23. Distribution of clinical complexity and utilisation

Identify skewness and concentrations at zero.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, feature in zip(axes.flat, clinical_features):
    sns.histplot(data=df, x=feature, discrete=True, ax=ax, color="teal")
    ax.set_title(feature.replace("_", " ").title())
    ax.set_xlabel(feature.replace("_", " ").title())
    ax.set_ylabel("Number of Encounters")
fig.suptitle("Clinical Complexity and Previous Healthcare Utilisation", fontsize=16)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### 24. Care intensity by recorded clinical complexity

Compare median observed care across recorded diagnosis counts. This is not yet a fitted expected-care benchmark.

In [ ]:
complexity_summary = df.groupby("number_diagnoses")[care_features].median()
display(complexity_summary)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, feature in zip(axes.flat, care_features):
    ax.plot(complexity_summary.index, complexity_summary[feature], marker="o", color="darkblue")
    ax.set_title(f"{care_labels[feature]} by Number of Diagnoses")
    ax.set_xlabel("Number of Recorded Diagnoses")
    ax.set_ylabel(f"Median {care_labels[feature]}")
fig.suptitle("Relationship Between Clinical Complexity and Care Intensity", fontsize=16)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### 25. Previous inpatient visits and readmission

Group prior-year inpatient visits into 0, 1, 2–3 and 4+. Zero does not mean no lifetime admission history.

In [ ]:
df["prior_inpatient_group"] = pd.cut(df["number_inpatient"], bins=[-1, 0, 1, 3, np.inf], labels=["0", "1", "2-3", "4+"])
prior_readmission = df.groupby("prior_inpatient_group", observed=True)["readmitted_30"].agg(
    Encounters="count", Early_Readmissions="sum", Rate="mean")
prior_readmission["Rate"] *= 100
prior_readmission = prior_readmission.rename(columns={"Rate": "Readmission Rate (%)"})
display(prior_readmission.round(2))
plt.figure(figsize=(10, 6))
prior_readmission["Readmission Rate (%)"].plot(kind="bar", color="purple", edgecolor="black")
plt.title("30-Day Readmission Rate by Previous Inpatient Visits")
plt.xlabel("Previous-Year Inpatient Visits")
plt.ylabel("30-Day Readmission Rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Phase 6 — Laboratory and medication analysis

### 26. HbA1c and maximum glucose results

Keep the recorded not-tested category (`None`) distinct from missing or unrecorded values.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, feature in zip(axes, ["A1Cresult", "max_glu_serum"]):
    counts = df[feature].fillna("Missing / Unrecorded").value_counts()
    print(feature)
    display(counts)
    counts.plot(kind="bar", ax=ax, color="steelblue")
    ax.set_title(f"Distribution of {feature}")
    ax.set_xlabel("Recorded Result")
    ax.set_ylabel("Number of Encounters")
    ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

### 27. Medication changes and readmission

Compare unadjusted readmission proportions by recorded medication change status. This is not a treatment-effect estimate.

In [ ]:
medication_readmission = pd.crosstab(df["change"], df["readmitted_30"], normalize="index").reindex(columns=[0, 1], fill_value=0) * 100
medication_readmission.columns = ["No Early Readmission", "Early Readmission"]
display(medication_readmission.round(2))
medication_readmission.plot(kind="bar", stacked=True, figsize=(10, 6), color=["seagreen", "crimson"])
plt.title("Readmission Percentage by Medication Change")
plt.xlabel("Medication Change Status")
plt.ylabel("Percentage of Encounters")
plt.xticks(rotation=0)
plt.legend(title="Readmission")
plt.tight_layout()
plt.show()

## Phase 7 — Multivariate analysis

### 28. Spearman correlation heatmap

Explore monotonic relationships among care intensity, clinical complexity, utilisation and readmission. Correlations are unadjusted and do not establish causation.

In [ ]:
correlation_features = care_features + clinical_features + ["readmitted_30"]
correlation_matrix = df[correlation_features].corr(method="spearman")
plt.figure(figsize=(13, 10))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, linewidths=0.5, square=True)
plt.title("Spearman Correlation: Care Intensity, Clinical Complexity and Readmission")
plt.tight_layout()
plt.show()

### 29. Relationships between care-intensity dimensions

Use a reproducible random sample of up to 3,000 encounters to reduce plotting time. The sample is for visualisation only.

In [ ]:
sample_df = df.sample(n=min(3000, len(df)), random_state=42)
plot = sns.pairplot(data=sample_df[care_features + ["readmitted_30"]], hue="readmitted_30",
                   hue_order=[0, 1], palette=["seagreen", "crimson"], diag_kind="hist",
                   corner=True, plot_kws={"alpha": 0.4})
plot.fig.suptitle("Relationships Between Care-Intensity Dimensions", y=1.02)
plt.show()

## EDA findings and next steps

Complete these observations after running the notebook on the actual dataset. Do not invent numerical findings.

| Area | Observation to record |
|---|---|
| Data quality | Dataset size, missing-value patterns, duplicate IDs and repeat encounters |
| Demographics | Representation across age, gender and race; small subgroup sizes |
| Readmission | Positive class prevalence and class imbalance |
| Care intensity | Typical ranges, distribution shapes and unusual values |
| Clinical complexity | How observed care changes with diagnosis counts and prior utilisation |
| RQ1 | Unadjusted care/readmission patterns to investigate using adjusted analysis |
| RQ2 | Candidate features and evaluation metrics; no predictive claim yet |

**Next stage:** `cleaning/` — define the eligible discharge cohort (including how to handle deaths, transfers and other discharge dispositions), preserve patient IDs for grouped splitting, and document missingness and encoding decisions. Fit preprocessing on training data only. Expected-care estimates for training records must be out-of-fold with patient-grouped folds; apply trained models to held-out patients. Use standardised residuals for the four discordance dimensions and compare identical classifier splits across the three agreed feature sets.

Expected care means the statistically typical recorded care conditional on chosen predictors, not medically recommended care. Neither this EDA nor the historical dataset can prove premature discharge, inappropriate treatment or actual cost savings.